# 52 Fusion Test

Score the fusion model (saved by notebook 51) and every baseline once on the 20 fusion test videos. All choices (C, temperature, inputs) were made on train + val in notebook 50. No setting is changed after this notebook.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

from woofdecoder.config import REPO_ROOT
from woofdecoder.fusion import FusionModel
from woofdecoder.fusion.fusers import FUSION_ENCODER, FUSION_LAYER, head_to_categories
from woofdecoder.taxonomy import CATEGORIES
from wooftrainer.audio_trainer.audio_heads import balanced_weights, fit_linear_head
from wooftrainer.fusion_trainer.branch_outputs import load_branch_outputs
from wooftrainer.fusion_trainer.train import (
    BRANCH_NAMES,
    branch_probs,
    compare_methods,
    features_from_cache,
    raw_average,
    score,
    video_weights,
)

## Helper methods

In [ ]:
pd.set_option("display.width", 200)
FUSION_DIR = REPO_ROOT / "models" / "fusion"

def video_predictions(probs, rows):
    # Mean probabilities per video -> (true categories, predicted categories)
    frame = pd.DataFrame(probs, columns=CATEGORIES)
    frame["video_id"] = rows["video_id"].to_numpy()
    frame["category"] = rows["category"].to_numpy()
    means = frame.groupby("video_id").agg({**dict.fromkeys(CATEGORIES, "mean"), "category": "first"})
    return means["category"].to_numpy(), means[CATEGORIES].idxmax(axis=1).to_numpy()

## Check model

In [ ]:
segments, videos = load_branch_outputs()
_, learned, trainval = compare_methods(segments)

saved_model = FusionModel.load(FUSION_DIR)
fusion = learned["fusion learned"]

print("choices (from train + val):")

for name, method in learned.items():
    print(f"  {name:<15} C = {method.C:<6} T = {method.temperature:.3f}")
    
print("\nmatches the saved model's temperature:", np.isclose(fusion.temperature, saved_model.head.temperature))

choices (from train + val):
  audio learned   C = 1.0    T = 1.165
  body learned    C = 1.0    T = 1.655
  scene learned   C = 0.1    T = 1.054
  fusion learned  C = 1.0    T = 1.373

matches the saved model's temperature: True


## Test, scored once

In [4]:
test = segments[segments["split"] == "test"].reset_index(drop=True)
no_segment = (videos[videos["split"] == "test"]["segments_kept"] == 0).sum()
print(f"test: {len(test)} segments, {test['video_id'].nunique()} videos with a segment, "
      f"{no_segment} videos with none\n")

tv_labels = trainval["category"].to_numpy()
tv_weights = balanced_weights(tv_labels, video_weights(trainval))
tv_features = features_from_cache(trainval)
test_features = features_from_cache(test)
test_weights = balanced_weights(test["category"].to_numpy(), video_weights(test))

# Every method: fitted on all of train + val with its chosen C and T, scored on test.
test_probs = {}
for name in BRANCH_NAMES:
    test_probs[f"{name} raw"] = branch_probs(test, name)
test_probs["average raw"] = raw_average(test)
for name, method in learned.items():
    head = fit_linear_head(
        tv_features[:, method.columns], tv_labels, tv_weights, method.C, FUSION_LAYER, FUSION_ENCODER
    )
    head.temperature = method.temperature
    test_probs[name] = head_to_categories(
        head.predict_proba(test_features[:, method.columns]), head.classes
    )

# The saved model must give exactly what "fusion learned" gives here.
saved = head_to_categories(saved_model.head.predict_proba(test_features), saved_model.head.classes)
print("saved model matches:", np.allclose(saved, test_probs["fusion learned"]), "\n")

rows = []
for name, probs in test_probs.items():
    rows.append({"method": name, **score(probs, test, test_weights)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

test: 111 segments, 18 videos with a segment, 2 videos with none

saved model matches: True 

        method  segment_log_loss  segment_macro_f1  video_macro_f1  video_accuracy
     audio raw             1.518             0.410           0.371           0.389
      body raw             2.177             0.300           0.291           0.278
     scene raw             1.667             0.427           0.508           0.611
   average raw             1.386             0.418           0.549           0.556
 audio learned             1.226             0.477           0.538           0.611
  body learned             1.524             0.353           0.280           0.278
 scene learned             1.075             0.588           0.628           0.611
fusion learned             0.834             0.581           0.687           0.667


## Uncertainty and the confusion matrix

In [5]:
truth, fusion_pred = video_predictions(test_probs["fusion learned"], test)
_, audio_pred = video_predictions(test_probs["audio learned"], test)

rng = np.random.default_rng(0)
f1_fusion = []
f1_diff = []
for _ in range(2000):
    index = rng.integers(0, len(truth), len(truth))
    a = f1_score(truth[index], fusion_pred[index], labels=CATEGORIES, average="macro", zero_division=0)
    b = f1_score(truth[index], audio_pred[index], labels=CATEGORIES, average="macro", zero_division=0)
    f1_fusion.append(a)
    f1_diff.append(a - b)
f1_diff = np.array(f1_diff)

print("fusion video macro-F1, 95% CI:", np.percentile(f1_fusion, [2.5, 97.5]).round(3))
print("fusion − audio learned, 95% CI:", np.percentile(f1_diff, [2.5, 97.5]).round(3),
      f"| fusion ahead in {(f1_diff > 0).mean():.0%} of resamples\n")
print(pd.crosstab(truth, fusion_pred, rownames=["true"], colnames=["fusion (test videos)"]).to_string())

fusion video macro-F1, 95% CI: [0.362 0.866]
fusion − audio learned, 95% CI: [-0.122  0.344] | fusion ahead in 82% of resamples

fusion (test videos)  alert_aggressive  attention_seeking  distressed_lonely  other  playful_excited
true                                                                                                
alert_aggressive                     3                  0                  0      0                1
attention_seeking                    0                  3                  0      0                0
distressed_lonely                    0                  2                  1      0                0
other                                0                  0                  0      2                0
playful_excited                      2                  1                  0      0                3
